# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step template for loading, exploring, and processing the [FAIR\^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and access its structure using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print title and description
print(f"Dataset Name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, and the fields of each record set.

In [ ]:
# List all record sets and their fields by @id
print("Record sets in the dataset:")
rs_metadata = dataset.metadata.record_sets
record_set_ids = []
for rs in rs_metadata:
    rs_id = rs['@id']
    record_set_ids.append(rs_id)
    name = rs.get('name', rs_id)
    print(f"  - {rs_id}: {name}")
    print("    Fields and columns (@id):")
    for f in rs['fields']:
        fid = f['@id']
        col = f['columns'] if isinstance(f['columns'], list) else [f['columns']]
        print(f"      * Field @id: {fid}")
        for c in col:
            print(f"        - Column @id: {c['@id']}")

Below, we print a few example records for each record set using their `@id`.

In [ ]:
# Print up to 2 records for each record set
for rsid in record_set_ids:
    print(f"\nExample records for record set: {rsid}")
    try:
        for idx, rec in enumerate(dataset.records(record_set=rsid)):
            print(json.dumps(rec, indent=2))
            if idx >= 1:
                break
    except Exception as e:
        print(f"  Could not load records: {e}")

## 3. Data Extraction
Load records from each record set as DataFrames. Use the record set and field `@id`s identified above.

In [ ]:
# Extract data from each record set into a DataFrame
dataframes = {}

for rsid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        if records:
            dataframes[rsid] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records from record set {rsid}")
        else:
            print(f"No records found in record set {rsid}")
    except Exception as e:
        print(f"Failed to load records from {rsid}: {e}")

Let's inspect the columns of a primary record set. Here, we choose the main table (likely the largest record set, usually with 'main' or similar in the `@id`).

In [ ]:
# Pick a primary record set, e.g., 'cr:main' if present (change to actual main set in this dataset)
main_rs = record_set_ids[0]
print(f"Columns available in {main_rs}:")
print(dataframes[main_rs].columns.tolist())
dataframes[main_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply common EDA steps: filtering, normalization, and grouping. We select a numeric field from the columns (e.g., a column with counts or measurements).

In [ ]:
# Automatically select a numeric field (first numeric column found)
import numpy as np

df = dataframes[main_rs]

# Find numeric columns
numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_fields:
    # Try numeric-looking columns (object types with strings that can be converted)
    possible_numeric = []
    for col in df.columns:
        try:
            _ = pd.to_numeric(df[col].dropna().iloc[:10])
            possible_numeric.append(col)
        except:
            pass
    numeric_fields = possible_numeric

if numeric_fields:
    numeric_field = numeric_fields[0]
else:
    raise ValueError("No numeric field found in the data!")

print(f"Using field '{numeric_field}' as numeric field for filtering/normalizing.")

# Convert the field to numeric if necessary
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

threshold = df[numeric_field].mean()
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} out of {len(df)} records.")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by first non-numeric field
group_fields = [col for col in df.columns if col != numeric_field and df[col].dtype == 'object']
if group_fields:
    group_field = group_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and the group means.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.hist(df[numeric_field].dropna(), bins=30, alpha=0.7, color='teal')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.title(f'Distribution of {numeric_field}')
plt.show()

if 'grouped_df' in locals():
    plt.figure(figsize=(12,4))
    grouped_df = grouped_df.sort_values(numeric_field, ascending=False).head(20)  # Show top 20 groups
    plt.bar(grouped_df[group_field].astype(str), grouped_df[numeric_field], color='orange')
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.title(f"Mean {numeric_field} by {group_field} (Top 20 groups)")
    plt.xticks(rotation=60)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook showcased how to load, explore, and process the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) with `mlcroissant`. We inspected the dataset's structure, loaded data into DataFrames, selected numeric fields for filtering and normalization, grouped records for summary statistics, and visualized key data distributions. 

Further work can be performed by leveraging the dataset's rich feature set for domain-specific analyses such as differential protein abundance, modification pattern exploration, and statistical modeling relevant to proteomics and extracellular vesicle research.